In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import scipy.ndimage as ndimage
import glob, os, shutil, cv2, json, sys
from tqdm import tqdm
from pathlib import Path

In [2]:
DATASET01_PATH = '../dataset_wu'
DATASET02_PATH = '../marlim_opt'

In [3]:
BASE_PATH = f'../Dataset/{Path().resolve().name}'
BASE_PATH

'../Dataset/dataset_wu_mar'

In [4]:
def getFiles(path, limit=None, shuffle=False):
    target = sorted([os.path.abspath(f) for f in glob.glob(os.path.join(path, '*'))])
    if shuffle:
        np.random.shuffle(target)     
    return target[:limit]

def setFolder(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)


OPTIONS = json.loads(open('../../Task/info.json', 'r').read())
OPTIONS['dataset'] = BASE_PATH.split('/')[-1].strip()
OPTIONS

{'network': 'resaceunet',
 'dataset': 'dataset_wu_mar',
 'img_size': None,
 'lr': 0.001,
 'loss': 'dice_focal',
 'batch_size': 2,
 'scheduler': 'plateau',
 'dropout': 0.1,
 'num_filters': 32}

In [5]:
with open('../../Task/info.json', 'w', encoding='utf-8') as file:
    json.dump(OPTIONS, file, ensure_ascii=False, indent=4) 

In [6]:
d1_images = getFiles(f'{DATASET01_PATH}/images')
d1_masks  = getFiles(f'{DATASET01_PATH}/masks')

print(len(d1_images))
d1_images[:5]

220


['/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/0.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/1.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/10.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/100.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/101.npy']

In [7]:
d2_images = getFiles(f'{DATASET02_PATH}/images')
d2_masks  = getFiles(f'{DATASET02_PATH}/masks')
print(len(d2_images))
d2_images[:5]

220


['/home/grva-mint/Projects/Falhas/Dataset/marlim_opt/images/img_0000.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/marlim_opt/images/img_0001.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/marlim_opt/images/img_0002.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/marlim_opt/images/img_0003.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/marlim_opt/images/img_0004.npy']

In [8]:
df = pd.DataFrame()
df['img_path']  = d1_images + d2_images
df['mask_path'] = d1_masks  + d2_masks
df

,img_path,mask_path
0,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
1,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
2,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
3,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
4,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
...,...,...
435,/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
436,/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
437,/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
438,/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...


In [9]:
info = []

for img_path, mask_path in tqdm(zip(df.img_path, df.mask_path)):
    img  = np.load(img_path)
    mask = np.load(mask_path)
    data = dict()

    data['id'] = Path(img_path).name
    data['img_min']  = np.min(img)
    data['img_max']  = np.max(img)
    data['img_mean'] = np.mean(img)
    data['img_std']  = np.std(img)

    data['msk_min'] = np.min(mask)
    data['msk_max'] = np.max(mask)
    data['shape']   = img.shape
    data['img_path']  = img_path
    data['mask_path'] = mask_path
    info.append(data)


df = pd.DataFrame(info)
print(df.img_path.iloc[2])
df

440it [00:02, 170.83it/s]

/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/10.npy


,id,img_min,img_max,img_mean,img_std,msk_min,msk_max,shape,img_path,mask_path
0,0.npy,0.0,1.0,0.497236,0.211666,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
1,1.npy,0.0,1.0,0.496481,0.184599,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
2,10.npy,0.0,1.0,0.496158,0.185499,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
3,100.npy,0.0,1.0,0.496551,0.203066,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
4,101.npy,0.0,1.0,0.497007,0.210493,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
...,...,...,...,...,...,...,...,...,...,...
435,img_0215.npy,0.0,1.0,0.500000,0.082691,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
436,img_0216.npy,0.0,1.0,0.502823,0.215750,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
437,img_0217.npy,0.0,1.0,0.508539,0.237145,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
438,img_0218.npy,0.0,1.0,0.501193,0.200873,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...


In [10]:
df.to_csv('../DataBase.csv', index=None)
df

,id,img_min,img_max,img_mean,img_std,msk_min,msk_max,shape,img_path,mask_path
0,0.npy,0.0,1.0,0.497236,0.211666,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
1,1.npy,0.0,1.0,0.496481,0.184599,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
2,10.npy,0.0,1.0,0.496158,0.185499,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
3,100.npy,0.0,1.0,0.496551,0.203066,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
4,101.npy,0.0,1.0,0.497007,0.210493,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
...,...,...,...,...,...,...,...,...,...,...
435,img_0215.npy,0.0,1.0,0.500000,0.082691,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
436,img_0216.npy,0.0,1.0,0.502823,0.215750,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
437,img_0217.npy,0.0,1.0,0.508539,0.237145,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
438,img_0218.npy,0.0,1.0,0.501193,0.200873,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/marlim...,/home/grva-mint/Projects/Falhas/Dataset/marlim...
